In [ ]:
from IPython.display import HTML
HTML(open('../style.css', 'r').read())

# Computing the Reflexive and Transitive Closure of a Relation

In the lecture notes, the language $L(F)$ accepted by an <span style="font-variant:small-caps;">Nfa</span> $F$ is defined with the help of the
*reflexive and transitive closure* $\leadsto^*$ of the relation $\leadsto$ that describes a single step of the
<span style="font-variant:small-caps;">Nfa</span>.  In this notebook we show how the reflexive and transitive closure $R^*$ of a binary relation $R$ on a
finite set $M$ can be computed.  As an application, we solve the puzzle of the *missionaries and the infidels*
who have to cross a river.  We describe the crossings of the river as a binary relation on the set of states of
the puzzle.  Then the puzzle is solvable if and only if the pair consisting of the start state and the goal state
is an element of the reflexive and transitive closure of this relation.

## Type Checking

The functions in this notebook carry *type annotations*.  *Python* itself ignores these annotations, but the
type checker [*basedpyright*](https://docs.basedpyright.com) can use them to find errors before the program is
run.  In *JupyterLab*, the extension *jupyterlab-lsp* runs *basedpyright* in the background and underlines
type errors while you type.  On the command line, the command
```
basedpyright ReflexiveTransitiveClosure.ipynb
```
checks the whole notebook.  The settings of the type checker are stored in the file `pyrightconfig.json` in the
directory `Python`.

Both packages, `basedpyright` and `jupyterlab-lsp`, are installed by the script `fl.sh`.  Start `jupyter lab` in the directory `Python`, so that the settings in `pyrightconfig.json` are used.

## Binary Relations

Given a set $M$, a *binary relation* $R$ on $M$ is a subset of the cartesian product $M \times M$, i.e. we have
$$ R \subseteq M \times M. $$
A binary relation can be interpreted as a *directed graph*: The elements of $M$ are the nodes of the graph and
there is an edge from the node $x$ to the node $y$ if and only if $\langle x, y \rangle \in R$.

In *Python*, we represent a binary relation as a set of pairs.  The elements of $M$ can be arbitrary values,
but they have to be *hashable*, since they are stored in sets.  Hence the type `Relation` has a type parameter
`T`, which is the type of the elements of $M$.

In [ ]:
type Relation[T] = set[tuple[T, T]]

## Displaying Relations as Graphs

In order to visualize a binary relation $R$ on a set $M$, we use the package
[`graphviz`](https://graphviz.readthedocs.io).

In [ ]:
import graphviz as gv

The function `relation2dot` turns every element of $M$ into a node and every pair $\langle x, y \rangle \in R$
into an edge from $x$ to $y$.  Many relations, e.g. the relation that we use below to model the puzzle, are
*symmetric*: If $\langle x, y \rangle \in R$, then $\langle y, x \rangle \in R$, too.  To keep the graph
readable, the two edges $\langle x, y \rangle$ and $\langle y, x \rangle$ are drawn as a single edge with arrow
heads at both ends.  The set `Drawn` contains the pairs that have already been drawn.

The optional argument `Path` is a tuple of elements of $M$.  The nodes of this path and the edges between
consecutive nodes of this path are drawn in red.  Below, we use this argument to display the solution of the
puzzle.

The optional argument `engine` selects the layout engine of `graphviz`.  The graphs are drawn from left to right:
- The default engine `dot` arranges the nodes in layers.  As the attribute `rankdir` is set to `'LR'`, most edges
  point from left to right.  This works well for the small examples below.
- The graph of the puzzle looks better with the engine `neato`, which uses a *spring model*: Every edge pulls its
  two nodes together like a spring.  The attribute `len` specifies the preferred length of the edges.  The
  orientation of a drawing produced by `neato` is arbitrary.  Therefore, the attribute `normalize` rotates the
  drawing such that the first edge points from left to right.

The attribute `rankdir` is only used by `dot`, while the attributes `len` and `normalize` are only used by `neato`.
Every engine ignores the attributes that it does not use.

As the order in which the elements of a set are enumerated is not specified, the nodes and edges are sorted
according to their string representation.  This way, the layout of the graph does not change when the notebook is
executed again.

**Function `relation2dot(M, R, Path, engine)`**
- *Input:* `M` is a set, `R` is a binary relation on `M`, `Path` is a tuple of elements of `M`, and `engine` is
  the name of a layout engine of `graphviz`.  The arguments `Path` and `engine` are optional and default to the empty
  tuple and to `'dot'`, respectively.
- *Output:* A `graphviz` graph of `R`.  The nodes and edges of `Path` are drawn in red.

In [ ]:
def relation2dot[T](M: set[T], R: Relation[T], Path: tuple[T, ...] = (), engine: str = 'dot') -> gv.Digraph:
    PathEdges = { (Path[i], Path[i+1]) for i in range(len(Path) - 1) }
    dot = gv.Digraph(engine=engine)
    dot.graph_attr['rankdir']   = 'LR'
    dot.graph_attr['normalize'] = '0'
    dot.edge_attr['len']        = '1.8'
    for x in sorted(M, key=str):
        if x in Path:
            dot.node(str(x), color='red', fontcolor='red')
        else:
            dot.node(str(x))
    Drawn: Relation[T] = set()
    for (x, y) in sorted(R, key=str):
        if (y, x) in Drawn:
            continue
        attributes: dict[str, str] = {}
        if (y, x) in R and x != y:
            attributes['dir'] = 'both'
        if (x, y) in PathEdges or (y, x) in PathEdges:
            attributes['color']    = 'red'
            attributes['penwidth'] = '2'
        dot.edge(str(x), str(y), **attributes)
        Drawn.add((x, y))
    return dot

As a first example, we take the set $M = \{1, 2, 3, 4\}$ and the relation
$R = \bigl\{ \langle 1, 2 \rangle, \langle 2, 3 \rangle, \langle 3, 4 \rangle \bigr\}$.

In [ ]:
M = { 1, 2, 3, 4 }
R = { (1, 2), (2, 3), (3, 4) }

In [ ]:
relation2dot(M, R)

## The Reflexive and Transitive Closure

The *reflexive and transitive closure* $R^*$ of a binary relation $R$ on a set $M$ is the smallest binary
relation $T$ on $M$ such that
1. $R \subseteq T$, i.e. $T$ contains $R$,
2. $T$ is *reflexive* on $M$, i.e. we have $\forall x \in M: \langle x, x \rangle \in T$, and
3. $T$ is *transitive*, i.e. we have
   $\forall x, y, z \in M: \bigl(\langle x, y \rangle \in T \wedge \langle y, z \rangle \in T \rightarrow \langle x, z \rangle \in T\bigr)$.

If we interpret $R$ as a directed graph, then $\langle x, y \rangle \in R^*$ holds if and only if there is a
*path* from $x$ to $y$, i.e. if there is a natural number $n$ and a list $[x_0, x_1, \cdots, x_n]$ of elements of
$M$ such that
$$ x_0 = x, \quad x_n = y, \quad \mbox{and} \quad \langle x_{i-1}, x_i \rangle \in R \quad \mbox{for all } i \in \{1, \cdots, n\}. $$
The number $n$ is the *length* of the path.  The case $n = 0$ is allowed: The list $[x]$ is a path of length $0$
from $x$ to $x$.  This is the reason why $R^*$ is reflexive.

In order to compute $R^*$ we first have to define two auxiliary functions.

### The Identity Relation

The *identity relation* on the set $M$ is defined as
$$ \texttt{id}_M := \bigl\{ \langle x, x \rangle \bigm| x \in M \bigr\}. $$
A binary relation $T$ on $M$ is reflexive if and only if $\texttt{id}_M \subseteq T$.

**Function `identity(M)`**
- *Input:* `M` is a set.
- *Output:* The identity relation $\texttt{id}_M$ on `M`.

In [ ]:
def identity[T](M: set[T]) -> Relation[T]:
    return { (x, x) for x in M }

In [ ]:
identity({ 1, 2, 3 })

### The Composition of Relations

The *composition* $R_1 \circ R_2$ of two binary relations $R_1$ and $R_2$ is defined as
$$ R_1 \circ R_2 := \bigl\{ \langle x, z \rangle \bigm| \exists y: \langle x, y \rangle \in R_1 \wedge \langle y, z \rangle \in R_2 \bigr\}. $$
Hence, the composition $R_1 \circ R_2$ contains an edge from $x$ to $z$ if we can get from $x$ to $z$ by first
following an edge of $R_1$ and then following an edge of $R_2$.  A binary relation $T$ is transitive if and only if
$T \circ T \subseteq T$.

**Function `compose(R1, R2)`**
- *Input:* `R1` and `R2` are binary relations on the same set.
- *Output:* The composition $R_1 \circ R_2$.

In [ ]:
def compose[T](R1: Relation[T], R2: Relation[T]) -> Relation[T]:
    return { (x, z) for (x, y1) in R1
                    for (y2, z) in R2
                    if  y1 == y2
           }

In [ ]:
compose({ (1, 2), (2, 3) }, { (2, 4), (3, 5) })

### Computing $R^*$ with a Fixed-Point Algorithm

The reflexive and transitive closure $R^*$ is computed with a *fixed-point algorithm* that is similar to the
computation of the $\varepsilon$-closure in the notebook `01-NFA-2-DFA.ipynb`.  We define a sequence of
relations $(T_n)_{n\in\mathbb{N}}$ inductively:
- $T_0 := \texttt{id}_M \cup R$,
- $T_{n+1} := T_n \cup T_n \circ T_n$.

By induction on $n$ it can be shown that $\langle x, y \rangle \in T_n$ holds if and only if there is a path of
length at most $2^n$ from $x$ to $y$.  This sequence of relations has the following properties:
- $T_0 \subseteq T_1 \subseteq T_2 \subseteq \cdots$ and all relations $T_n$ are subsets of the finite set
  $M \times M$.  Hence this sequence cannot grow forever: There is a $k \in \mathbb{N}$ such that
  $T_{k+1} = T_k$.  Then $T_k \circ T_k \subseteq T_k$ and therefore $T_k$ is transitive.  As
  $\texttt{id}_M \cup R \subseteq T_k$, the relation $T_k$ is also reflexive and contains $R$.
- If $\langle x, y \rangle \in T_k$, then there is a path from $x$ to $y$.  Therefore, $\langle x, y \rangle$ is an
  element of every reflexive and transitive relation that contains $R$.  Hence $T_k$ is the *smallest* such
  relation and we have $T_k = R^*$.

If there is a path from $x$ to $y$, then there is also a path from $x$ to $y$ that does not visit a node twice.
Such a path has a length of at most $|M| - 1$.  Therefore, $T_n = R^*$ as soon as $2^n \geq |M| - 1$.  Hence the
number of iterations of the algorithm is only logarithmic in the number of elements of $M$.

The function `closure` implements this algorithm.  The variable `Result` contains the relation $T_n$.  In every
iteration we compute $T_n \circ T_n$.  If this composition does not contain any new pairs, then $T_n$ is transitive
and we return it.

**Function `closure(M, R)`**
- *Input:* `M` is a finite set and `R` is a binary relation on `M`.
- *Output:* The reflexive and transitive closure $R^*$ of `R`.

In [ ]:
def closure[T](M: set[T], R: Relation[T]) -> Relation[T]:
    Result = identity(M) | R
    while True:
        NewPairs = compose(Result, Result)
        if NewPairs <= Result:
            return Result
        Result |= NewPairs

Let us compute the reflexive and transitive closure of the relation $R$ defined above.

In [ ]:
RStar = closure(M, R)
RStar

In the graph of $R^*$, every node has a loop, since $R^*$ is reflexive, and there is an edge from every node to all
nodes with a bigger number, since $R^*$ is transitive.

In [ ]:
relation2dot(M, RStar)

<hr style="height:5px;background-color:blue">

## Application: The Missionaries and the Infidels

Three missionaries and three infidels have to cross a river from its left bank to its right bank.  They have a
single boat that can carry at most two persons.  The boat cannot cross the river on its own: At least one person
has to row it.  The missionaries are in danger if, at any time, the infidels outnumber the missionaries on one of
the banks of the river, unless there are no missionaries on this bank at all.  How can all six persons get to
the right bank without the missionaries ever being in danger?

In the classic version of this puzzle the infidels are cannibals.  This is why we use the letter $c$ for the
number of infidels.

### Modelling the States

A state of the puzzle is represented as a triple $(m, c, b)$ where
- $m \in \{0, 1, 2, 3\}$ is the number of missionaries on the left bank,
- $c \in \{0, 1, 2, 3\}$ is the number of infidels on the left bank, and
- $b \in \{0, 1\}$ is the number of boats on the left bank.

Then there are $3 - m$ missionaries, $3 - c$ infidels, and $1 - b$ boats on the right bank.  In the start state
everybody is on the left bank, so the start state is $(3, 3, 1)$.  The goal state is $(0, 0, 0)$, as
everybody, including the boat, has to get to the right bank.

In [ ]:
type State = tuple[int, int, int]

The function `problem` checks whether the missionaries on one bank of the river are in danger.

**Function `problem(m, c)`**
- *Input:* `m` is the number of missionaries and `c` is the number of infidels on one bank of the river.
- *Output:* `True` if there is at least one missionary on this bank and the missionaries are outnumbered by the
  infidels, `False` otherwise.

In [ ]:
def problem(m: int, c: int) -> bool:
    return 0 < m < c

The function `noProblemAtAll` checks both banks of the river.

**Function `noProblemAtAll(m, c)`**
- *Input:* `m` is the number of missionaries and `c` is the number of infidels on the left bank of the river.
- *Output:* `True` if the missionaries are safe on both banks of the river, `False` otherwise.

In [ ]:
def noProblemAtAll(m: int, c: int) -> bool:
    return not problem(m, c) and not problem(3 - m, 3 - c)

The set `States` contains all states in which the missionaries are safe.  The other states are not needed, as the
puzzle must never reach them.

In [ ]:
States: set[State] = { (m, c, b) for m in range(4)
                                 for c in range(4)
                                 for b in range(2)
                                 if  noProblemAtAll(m, c)
                     }

In [ ]:
len(States)

### Modelling the Crossings as a Binary Relation

If the boat carries $\Delta m$ missionaries and $\Delta c$ infidels, then $1 \leq \Delta m + \Delta c \leq 2$,
since the boat needs somebody to row it and has room for at most two persons.  The set `BoatLoads` contains all
pairs $(\Delta m, \Delta c)$ that satisfy this condition.

In [ ]:
BoatLoads = { (dm, dc) for dm in range(3)
                       for dc in range(3)
                       if  1 <= dm + dc <= 2
            }
BoatLoads

Now we define the binary relation `Crossings` on the set `States`.  A pair $\langle s_1, s_2 \rangle$ is an element of
`Crossings` if a single crossing of the river leads from the state $s_1$ to the state $s_2$:
- If the boat is on the left bank, i.e. if $b = 1$, then the boat takes $\Delta m$ missionaries and $\Delta c$
  infidels to the right bank.  This leads from the state $(m, c, 1)$ to the state
  $(m - \Delta m, c - \Delta c, 0)$.
- If the boat is on the right bank, i.e. if $b = 0$, then the boat takes $\Delta m$ missionaries and $\Delta c$
  infidels to the left bank.  This leads from the state $(m, c, 0)$ to the state
  $(m + \Delta m, c + \Delta c, 1)$.

In both cases, the new state has to be an element of `States`.  This guarantees that the numbers of missionaries
and infidels are between $0$ and $3$ and that the missionaries are safe after the crossing.  Therefore, the
relation `Crossings` is defined as $\texttt{LeftToRight} \cup \texttt{RightToLeft}$ where
$$
\begin{array}{lcl}
\texttt{LeftToRight} & := & \bigl\{ \bigl\langle (m, c, 1), (m - \Delta m, c - \Delta c, 0) \bigr\rangle \bigm|
  (m, c, 1) \in \texttt{States} \wedge (\Delta m, \Delta c) \in \texttt{BoatLoads} \wedge
  (m - \Delta m, c - \Delta c, 0) \in \texttt{States} \bigr\}, \\[0.2cm]
\texttt{RightToLeft} & := & \bigl\{ \bigl\langle (m, c, 0), (m + \Delta m, c + \Delta c, 1) \bigr\rangle \bigm|
  (m, c, 0) \in \texttt{States} \wedge (\Delta m, \Delta c) \in \texttt{BoatLoads} \wedge
  (m + \Delta m, c + \Delta c, 1) \in \texttt{States} \bigr\}.
\end{array}
$$

In [ ]:
LeftToRight: Relation[State] = { ((m, c, 1), (m - dm, c - dc, 0))
                                 for (m, c, b) in States if b == 1
                                 for (dm, dc) in BoatLoads
                                 if  (m - dm, c - dc, 0) in States
                               }

In [ ]:
RightToLeft: Relation[State] = { ((m, c, 0), (m + dm, c + dc, 1))
                                 for (m, c, b) in States if b == 0
                                 for (dm, dc) in BoatLoads
                                 if  (m + dm, c + dc, 1) in States
                               }

In [ ]:
Crossings = LeftToRight | RightToLeft
len(Crossings)

Every crossing can be undone: The persons who have just crossed the river can take the boat back.  Hence the relation
`Crossings` is symmetric.

In [ ]:
Crossings == { (s2, s1) for (s1, s2) in Crossings }

The graph of the relation `Crossings` looks as follows.  As the relation is symmetric, every edge has arrow heads at
both ends.  The start state $(3, 3, 1)$ is on the left and the goal state $(0, 0, 0)$ is on the right.

In [ ]:
relation2dot(States, Crossings, engine='neato')

### Is the Puzzle Solvable?

The puzzle is solvable if and only if there is a sequence of crossings that leads from the start state to the goal
state, i.e. if and only if
$$ \bigl\langle (3, 3, 1), (0, 0, 0) \bigr\rangle \in \texttt{Crossings}^*. $$

In [ ]:
start: State = (3, 3, 1)
goal:  State = (0, 0, 0)

In [ ]:
CrossingsStar = closure(States, Crossings)

In [ ]:
(start, goal) in CrossingsStar

The relation $\texttt{Crossings}^*$ also tells us which states can be reached from the start state.

In [ ]:
Reachable = { s for (r, s) in CrossingsStar if r == start }
len(Reachable)

Four of the $20$ states can not be reached from the start state.  These are the isolated nodes in the graph shown
above: No crossing leads into these states.  For example, in the state $(0, 0, 1)$ the boat is on the left bank, but
all persons are on the right bank.  As the boat can not cross the river on its own, this state is impossible.

In [ ]:
States - Reachable

### Finding a Solution

The relation $\texttt{Crossings}^*$ tells us *that* the goal state can be reached, but it does not tell us *how* it
can be reached.  To find a solution of the puzzle, we need a path from the start state to the goal state.  To this
end, we modify the fixed-point algorithm: Instead of pairs $\langle x, y \rangle$, we now compute paths, i.e. tuples
$(x_0, x_1, \cdots, x_n)$ such that $x_0$ is the start state and $\langle x_{i-1}, x_i \rangle \in R$ holds for all
$i \in \{1, \cdots, n\}$.
- Initially, the set `Paths` contains only the path $(x_0)$ of length $0$.
- In every iteration, every path $(x_0, \cdots, x_n)$ in `Paths` is extended by every node $y$ such that
  $\langle x_n, y \rangle \in R$, provided $y$ does not already occur in the path.  We do not need paths that visit
  a node twice, since the part of the path between the two visits is a detour that can be cut out.  As there are
  only finitely many paths that do not visit a node twice, the algorithm terminates.
- After $n$ iterations, `Paths` contains all paths of length at most $n$ that start in the start state and do not
  visit a node twice.  Therefore, as soon as `Paths` contains a path that ends in the goal state, it contains all
  shortest paths from the start state to the goal state.  The function returns the set of these paths.
- If the set `Paths` does not grow anymore and still contains no path to the goal state, then the goal state can not
  be reached and the function returns the empty set.

In the worst case, the number of paths grows exponentially with the number of nodes.  For big graphs, the
search algorithms presented in the lecture on artificial intelligence, e.g. *breadth-first search*, are more
efficient.  For our puzzle, the number of paths is small.

**Function `findPaths(start, goal, R)`**
- *Input:* `start` and `goal` are elements of a set $M$ and `R` is a binary relation on $M$.
- *Output:* The set of all shortest paths from `start` to `goal`.  Every path is a tuple of elements of $M$.
  If there is no path from `start` to `goal`, the empty set is returned.

In [ ]:
def findPaths[T](start: T, goal: T, R: Relation[T]) -> set[tuple[T, ...]]:
    Paths: set[tuple[T, ...]] = { (start,) }
    while True:
        Found = { p for p in Paths if p[-1] == goal }
        if Found != set():
            return Found
        NewPaths = { p + (y,) for p in Paths
                              for (x, y) in R
                              if  p[-1] == x and y not in p
                   }
        if NewPaths <= Paths:
            return set()
        Paths |= NewPaths

In [ ]:
Solutions = findPaths(start, goal, Crossings)

There are four different solutions and each of them needs $11$ crossings.

In [ ]:
len(Solutions), { len(p) - 1 for p in Solutions }

We pick the smallest of these solutions with respect to the lexicographic order, so that the result does not
depend on the order in which the elements of the set `Solutions` are enumerated.

In [ ]:
solution = min(Solutions)
solution

The solution is shown in red in the graph of the relation `Crossings`.  The graph also shows why there are four
solutions:  Both at the beginning and at the end, there are two different ways to get around a small cycle.

In [ ]:
relation2dot(States, Crossings, solution, 'neato')

### Printing the Solution

Finally, we print the solution in a more readable form.  Every state is printed as a line that shows the persons
and the boat on both banks of the river.  The letter `M` denotes a missionary, the letter `C` denotes an infidel,
and the letter `B` denotes the boat.  Between two states, a line shows who crosses the river in which direction.

**Function `printState(s)`**
- *Input:* `s` is a state $(m, c, b)$.
- *Output:* None.  The function prints the persons and the boat on the left bank, the river, and the persons and the
  boat on the right bank.

In [ ]:
def printState(s: State) -> None:
    m, c, b = s
    left  = f'{"M" * m:<6}{"C" * c:<6}{"B" * b:<3}'
    right = f'{"M" * (3 - m):>6}{"C" * (3 - c):>6}{"B" * (1 - b):>3}'
    print(left + '    |~~~~~|    ' + right)

**Function `printBoat(s1, s2)`**
- *Input:* `s1` and `s2` are states such that a single crossing leads from `s1` to `s2`.
- *Output:* None.  The function prints the persons in the boat and the direction of the crossing.

In [ ]:
def printBoat(s1: State, s2: State) -> None:
    m1, c1, b1 = s1
    m2, c2, _  = s2
    load = 'M' * abs(m1 - m2) + ' ' + 'C' * abs(c1 - c2)
    if b1 == 1:
        print(19 * ' ' + f'> {load:^3} >')
    else:
        print(19 * ' ' + f'< {load:^3} <')

**Function `printPath(Path)`**
- *Input:* `Path` is a path, i.e. a non-empty tuple of states such that a single crossing leads from every state to
  the next state.
- *Output:* None.  The function prints all states of the path and the crossings between them.

In [ ]:
def printPath(Path: tuple[State, ...]) -> None:
    for i in range(len(Path) - 1):
        printState(Path[i])
        printBoat(Path[i], Path[i+1])
    printState(Path[-1])

In [ ]:
printPath(solution)